# S06-15B - Prepare Habitar 7.2 Graph

> Active notebook for Task 5. Input graph comes from `03_Build_Graph_Representation/graph_outputs`; the exported ML dataset is written to `05_Run_Node_Classification/ml_dataset`.


## Phase 0 — Overview and expected graph



### Encoding MSD-style edge and node features in a TopologicPy Graph

This notebook prepares a TopologicPy `Graph` so that `Graph.ExportToCSV(...)` can export edge and node features compatible with the feature schema used in the GraphSAGE node-classification workflow.

The expected input graph is a room graph:

- **Vertices** represent rooms.
- **Edges** represent access/connectivity relationships between rooms.
- Each **edge dictionary** already has a key named `door_type` with one of the following lower-case string values:
  - `"passage"`
  - `"door"`
  - `"entrance_door"`
- Each **vertex dictionary** already has a key named `room_type` with one of the nine lower-case room-type strings listed below.

The notebook then writes the required machine-learning features back into the dictionaries of the graph's edges and vertices.


## Phase 1 — Define the feature schema



### Feature schema

#### Edge features

The preprocessing code used for the original experiment encoded edge connectivity/access types as follows:

| `door_type` value | Integer class | One-hot edge feature |
|---|---:|---|
| `passage` | `0` | `[1, 0, 0]` |
| `door` | `1` | `[0, 1, 0]` |
| `entrance_door` | `2` | `[0, 0, 1]` |

For compatibility with the CSV feature naming convention, the edge dictionary receives:

```text
connectivity = 0/1/2
connectivity_0
connectivity_1
connectivity_2
```

#### Node features

Each vertex/room starts with a semantic `room_type`. The helper function derives:

1. a numeric room label, stored as `label`;
2. a project-specific four-class zoning label, stored as `zoning_type`;
3. a one-hot zoning vector, stored as:

```text
zoning_type_0
zoning_type_1
zoning_type_2
zoning_type_3
```

4. a three-value node connectivity vector, stored as:

```text
connectivity_0
connectivity_1
connectivity_2
```

The node connectivity vector is computed from the incident edges of each room. By default, the values are **per-node proportions**. For example, if a room has two incident `door` edges and one incident `passage` edge, its node connectivity feature becomes:

```text
connectivity_0 = 1/3
connectivity_1 = 2/3
connectivity_2 = 0
```



### Room labels and zoning assumptions

The nine room types follow the Modified Swiss Dwellings / CVAAD challenge class order:

| `room_type` | `label` |
|---|---:|
| `bedroom` | `0` |
| `livingroom` | `1` |
| `kitchen` | `2` |
| `dining` | `3` |
| `corridor` | `4` |
| `stairs` | `5` |
| `storeroom` | `6` |
| `bathroom` | `7` |
| `balcony` | `8` |

The four zoning classes are derived from the room type:

| Zoning class | One-hot index | Room types |
|---|---:|---|
| Private/static | `0` | `bedroom` |
| Living/dynamic | `1` | `livingroom`, `kitchen`, `dining`, `corridor` |
| Service/functional | `2` | `stairs`, `storeroom`, `bathroom` |
| Outdoor/semi-outdoor | `3` | `balcony` |

This zoning assignment matches the structure visible in the attached CSV dataset: bedrooms map to zoning type 0; livingroom, kitchen, dining, and corridor map to zoning type 1; stairs, storeroom, and bathroom map to zoning type 2; balcony maps to zoning type 3.


## Phase 2 — Prepare and encode graph features



### 2.0 Preparing the graph

Before running the helper function, make sure that:

1. Every graph edge has a dictionary entry named `door_type`.
2. The value of `door_type` is one of `"passage"`, `"door"`, or `"entrance_door"`.
3. Every graph vertex has a dictionary entry named `room_type`.
4. The value of `room_type` is one of the nine supported room types.
5. The graph topology is already correct: vertices represent rooms and edges represent room-to-room access/connectivity.

The helper function mutates the graph's dictionaries in place and also returns the graph for convenience.


### 2.1 Imports and canonical feature mappings


In [1]:

from topologicpy.Graph import Graph
from topologicpy.Edge import Edge
from topologicpy.Vertex import Vertex
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary


# -----------------------------------------------------------------------------
# Canonical feature mappings
# -----------------------------------------------------------------------------

DOOR_TYPE_TO_CONNECTIVITY = {
    "passage": 0,
    "door": 1,
    "entrance_door": 2,
    # Useful aliases. The original preprocessing notebook used "entrance".
    "entrance": 2,
    "entrance door": 2,
    "entrance-door": 2,
}

CONNECTIVITY_TO_DOOR_TYPE = {
    0: "passage",
    1: "door",
    2: "entrance_door",
}

ROOM_TYPE_TO_LABEL = {
    "bedroom": 0,
    "livingroom": 1,
    "living_room": 1,
    "living room": 1,
    "kitchen": 2,
    "dining": 3,
    "corridor": 4,
    "stairs": 5,
    "stair": 5,
    "staircase": 5,
    "storeroom": 6,
    "store_room": 6,
    "store room": 6,
    "storage": 6,
    "bathroom": 7,
    "balcony": 8,
}

# Project-specific zoning derived from room_type.
# 0: private/static, 1: living/dynamic, 2: service/functional, 3: outdoor/semi-outdoor
ROOM_TYPE_TO_ZONING = {
    "bedroom": 0,
    "livingroom": 1,
    "living_room": 1,
    "living room": 1,
    "kitchen": 1,
    "dining": 1,
    "corridor": 1,
    "stairs": 2,
    "stair": 2,
    "staircase": 2,
    "storeroom": 2,
    "store_room": 2,
    "store room": 2,
    "storage": 2,
    "bathroom": 2,
    "balcony": 3,
}

EDGE_FEATURE_KEYS = [
    "connectivity_0",
    "connectivity_1",
    "connectivity_2",
]

NODE_ZONING_FEATURE_KEYS = [
    "zoning_type_0",
    "zoning_type_1",
    "zoning_type_2",
    "zoning_type_3",
]

NODE_CONNECTIVITY_FEATURE_KEYS = [
    "connectivity_0",
    "connectivity_1",
    "connectivity_2",
]




C:\Users\Arq. David Agudelo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 2.2 Small internal utilities


In [2]:
# -----------------------------------------------------------------------------
# Small internal utilities
# -----------------------------------------------------------------------------

def _clean_string(value):
    """Normalises a dictionary string value for robust matching."""
    if value is None:
        return None
    return str(value).strip().lower().replace("-", "_")


def _one_hot(index, length):
    """Returns a one-hot vector of the requested length."""
    values = [0] * length
    values[int(index)] = 1
    return values


def _merge_dictionary(topology, keys, values):
    """Merges key/value pairs into the dictionary of a Topologic topology."""
    old_d = Topology.Dictionary(topology)
    new_d = Dictionary.ByKeysValues(keys, values)
    if old_d:
        merged_d = Dictionary.ByMergedDictionaries([old_d, new_d])
    else:
        merged_d = new_d
    Topology.SetDictionary(topology, merged_d)
    return topology


def _dictionary_value(topology, key, default=None):
    """Safely retrieves a value from a TopologicPy dictionary."""
    d = Topology.Dictionary(topology)
    return Dictionary.ValueAtKey(d, key, default)


def _vertex_xyz_key(vertex, mantissa=6):
    """Returns a rounded coordinate key for a vertex."""
    return (
        round(float(Vertex.X(vertex)), mantissa),
        round(float(Vertex.Y(vertex)), mantissa),
        round(float(Vertex.Z(vertex)), mantissa),
    )


def _find_vertex_index(vertex, vertices, xyz_index, mantissa=6, tolerance=0.0001):
    """
    Finds the index of a vertex in a list of graph vertices.

    The fast path uses rounded coordinates. If more than one vertex shares the
    same rounded coordinates, Topology.IsSame and Vertex.Distance are used as
    fallbacks.
    """
    key = _vertex_xyz_key(vertex, mantissa=mantissa)
    candidate_indices = xyz_index.get(key, [])

    for i in candidate_indices:
        try:
            if Topology.IsSame(vertex, vertices[i]):
                return i
        except Exception:
            pass

        try:
            if Vertex.Distance(vertex, vertices[i]) <= tolerance:
                return i
        except Exception:
            pass

    # Slow fallback if rounded coordinate lookup failed.
    for i, v in enumerate(vertices):
        try:
            if Topology.IsSame(vertex, v):
                return i
        except Exception:
            pass

        try:
            if Vertex.Distance(vertex, v) <= tolerance:
                return i
        except Exception:
            pass

    return None


def _edge_connectivity_index(edge, doorTypeKey="door_type"):
    """Returns the integer connectivity class 0, 1, or 2 for an edge."""
    door_type = _clean_string(_dictionary_value(edge, doorTypeKey, None))
    if door_type in DOOR_TYPE_TO_CONNECTIVITY:
        return DOOR_TYPE_TO_CONNECTIVITY[door_type]

    # Fallback: accept a pre-existing integer connectivity value.
    value = _dictionary_value(edge, "connectivity", None)
    if value is not None:
        try:
            value = int(value)
            if value in (0, 1, 2):
                return value
        except Exception:
            pass

    # Fallback: accept pre-existing one-hot feature keys.
    values = []
    for key in EDGE_FEATURE_KEYS:
        values.append(_dictionary_value(edge, key, 0))

    try:
        values = [float(v) for v in values]
        if sum(values) > 0:
            return int(values.index(max(values)))
    except Exception:
        pass

    return None




### 2.3 Public helper function


In [3]:
# -----------------------------------------------------------------------------
# Public helper function
# -----------------------------------------------------------------------------

def EncodeMSDGraphFeatures(graph,
                           roomTypeKey="room_type",
                           doorTypeKey="door_type",
                           overwrite=True,
                           normalizeNodeConnectivity=True,
                           writeReadableAliases=False,
                           mantissa=6,
                           tolerance=0.0001,
                           silent=False):
    """
    Encodes MSD-style edge and node features into a TopologicPy graph.

    Parameters
    ----------
    graph : topologic_core.Graph
        The input TopologicPy graph. Vertices are assumed to represent rooms.
        Edges are assumed to represent access/connectivity relationships.
    roomTypeKey : str , optional
        The vertex dictionary key containing the room type string. The value
        should be lower case and one of the supported room types. Default is
        "room_type".
    doorTypeKey : str , optional
        The edge dictionary key containing the access type string. Expected
        values are "passage", "door", or "entrance_door". Default is
        "door_type".
    overwrite : bool , optional
        If True, existing feature keys are overwritten. If False, existing
        values are preserved. Default is True.
    normalizeNodeConnectivity : bool , optional
        If True, node connectivity values are stored as proportions of each
        node's incident edge types. If False, raw incident counts are stored.
        Default is True.
    writeReadableAliases : bool , optional
        If True, also writes non-CSV alias keys "connectivity_0",
        "connectivity_1", and "connectivity_2" to the vertex dictionaries.
        For clean CSV export, leave this as False. Default is False.
    mantissa : int , optional
        The number of decimal places used for rounding coordinates and feature
        values. Default is 6.
    tolerance : float , optional
        The tolerance used when matching edge endpoints to graph vertices.
        Default is 0.0001.
    silent : bool , optional
        If True, warning messages are suppressed. Default is False.

    Returns
    -------
    topologic_core.Graph
        The input graph with updated edge and vertex dictionaries.

    Notes
    -----
    The canonical edge connectivity mapping is:
    - passage       -> 0 -> connectivity_0
    - door          -> 1 -> connectivity_1
    - entrance_door -> 2 -> connectivity_2

    The canonical room label mapping is:
    - bedroom    -> 0
    - livingroom -> 1
    - kitchen    -> 2
    - dining     -> 3
    - corridor   -> 4
    - stairs     -> 5
    - storeroom  -> 6
    - bathroom   -> 7
    - balcony    -> 8
    """

    if graph is None:
        if not silent:
            print("EncodeMSDGraphFeatures - Error: The input graph is None.")
        return None

    vertices = Graph.Vertices(graph)
    edges = Graph.Edges(graph)

    if not vertices:
        if not silent:
            print("EncodeMSDGraphFeatures - Error: The input graph has no vertices.")
        return graph

    if edges is None:
        edges = []

    edge_connectivity = _encode_edge_access_features(
        edges,
        doorTypeKey=doorTypeKey,
        overwrite=overwrite,
        silent=silent,
    )

    incident_counts = _build_incident_connectivity_counts(
        vertices,
        edges,
        edge_connectivity,
        mantissa=mantissa,
        tolerance=tolerance,
        silent=silent,
    )

    _encode_node_room_features(
        vertices,
        incident_counts,
        roomTypeKey=roomTypeKey,
        overwrite=overwrite,
        normalizeNodeConnectivity=normalizeNodeConnectivity,
        writeReadableAliases=writeReadableAliases,
        mantissa=mantissa,
        silent=silent,
    )

    return graph


### 2.4 Encode edge access and connectivity features


In [4]:
def _encode_edge_access_features(edges,
                                 doorTypeKey="door_type",
                                 overwrite=True,
                                 silent=False):
    """Encodes edge access types and returns their connectivity classes."""

    edge_connectivity = []

    for edge in edges:
        c_index = _edge_connectivity_index(edge, doorTypeKey=doorTypeKey)

        if c_index is None:
            edge_connectivity.append(None)
            if not silent:
                print("EncodeMSDGraphFeatures - Warning: An edge has no recognised door_type/connectivity. It will not contribute to node connectivity.")
            continue

        one_hot = _one_hot(c_index, 3)
        keys = ["connectivity"] + EDGE_FEATURE_KEYS
        values = [int(c_index)] + one_hot

        # Preserve/standardise the semantic door type as well.
        keys.append(doorTypeKey)
        values.append(CONNECTIVITY_TO_DOOR_TYPE[int(c_index)])

        if overwrite:
            _merge_dictionary(edge, keys, values)
        else:
            existing_keys = []
            existing_values = []
            for key, value in zip(keys, values):
                if _dictionary_value(edge, key, None) is None:
                    existing_keys.append(key)
                    existing_values.append(value)
            if existing_keys:
                _merge_dictionary(edge, existing_keys, existing_values)

        edge_connectivity.append(c_index)

    return edge_connectivity


### 2.5 Build the vertex lookup and incident counts


In [5]:
def _build_incident_connectivity_counts(vertices,
                                        edges,
                                        edge_connectivity,
                                        mantissa=6,
                                        tolerance=0.0001,
                                        silent=False):
    """Builds the vertex lookup and counts incident connectivity types."""

    xyz_index = {}
    for i, vertex in enumerate(vertices):
        key = _vertex_xyz_key(vertex, mantissa=mantissa)
        xyz_index.setdefault(key, []).append(i)

    incident_counts = [[0.0, 0.0, 0.0] for _ in vertices]

    for edge, c_index in zip(edges, edge_connectivity):
        if c_index is None:
            continue

        try:
            sv = Edge.StartVertex(edge)
            ev = Edge.EndVertex(edge)
        except Exception:
            if not silent:
                print("EncodeMSDGraphFeatures - Warning: Could not retrieve start/end vertices for an edge. Skipping it.")
            continue

        si = _find_vertex_index(sv, vertices, xyz_index, mantissa=mantissa, tolerance=tolerance)
        ei = _find_vertex_index(ev, vertices, xyz_index, mantissa=mantissa, tolerance=tolerance)

        if si is not None:
            incident_counts[si][c_index] += 1.0
        elif not silent:
            print("EncodeMSDGraphFeatures - Warning: Could not match an edge start vertex to the graph vertex list.")

        if ei is not None and ei != si:
            incident_counts[ei][c_index] += 1.0
        elif ei is None and not silent:
            print("EncodeMSDGraphFeatures - Warning: Could not match an edge end vertex to the graph vertex list.")

    return incident_counts


### 2.6 Encode node room, zoning, and connectivity features


In [6]:
def _encode_node_room_features(vertices,
                               incident_counts,
                               roomTypeKey="room_type",
                               overwrite=True,
                               normalizeNodeConnectivity=True,
                               writeReadableAliases=False,
                               mantissa=6,
                               silent=False):
    """Encodes room labels, zoning, and node connectivity features."""

    for i, vertex in enumerate(vertices):
        room_type_raw = _dictionary_value(vertex, roomTypeKey, None)
        room_type = _clean_string(room_type_raw)

        if room_type not in ROOM_TYPE_TO_LABEL:
            if not silent:
                print(f"EncodeMSDGraphFeatures - Warning: Unsupported room_type {room_type_raw!r}. This vertex will receive label=-1 and zero zoning features.")
            label = -1
            zoning_type = -1
            zoning_features = [0, 0, 0, 0]
        else:
            label = ROOM_TYPE_TO_LABEL[room_type]
            zoning_type = ROOM_TYPE_TO_ZONING[room_type]
            zoning_features = _one_hot(zoning_type, 4)

        connectivity_values = incident_counts[i]
        if normalizeNodeConnectivity:
            total = sum(connectivity_values)
            if total > 0:
                connectivity_values = [round(v / total, mantissa) for v in connectivity_values]
            else:
                connectivity_values = [0.0, 0.0, 0.0]
        else:
            connectivity_values = [round(v, mantissa) for v in connectivity_values]

        keys = [roomTypeKey, "label", "zoning_type"]
        values = [room_type if room_type is not None else room_type_raw, int(label), int(zoning_type)]

        keys += NODE_ZONING_FEATURE_KEYS
        values += zoning_features

        keys += NODE_CONNECTIVITY_FEATURE_KEYS
        values += connectivity_values

        if writeReadableAliases:
            keys += ["connectivity_0", "connectivity_1", "connectivity_2"]
            values += connectivity_values

        if overwrite:
            _merge_dictionary(vertex, keys, values)
        else:
            missing_keys = []
            missing_values = []
            for key, value in zip(keys, values):
                if _dictionary_value(vertex, key, None) is None:
                    missing_keys.append(key)
                    missing_values.append(value)
            if missing_keys:
                _merge_dictionary(vertex, missing_keys, missing_values)

    return vertices


In [7]:

def CheckMSDGraphPreparation(graph,
                             roomTypeKey="room_type",
                             doorTypeKey="door_type",
                             silent=False):
    """
    Checks whether a graph is ready for EncodeMSDGraphFeatures.

    Returns
    -------
    dict
        A summary dictionary containing counts and lists of unsupported values.
    """

    vertices = Graph.Vertices(graph) if graph else []
    edges = Graph.Edges(graph) if graph else []

    unsupported_room_types = []
    unsupported_door_types = []
    missing_room_type_count = 0
    missing_door_type_count = 0

    for vertex in vertices:
        value = _dictionary_value(vertex, roomTypeKey, None)
        clean = _clean_string(value)
        if clean is None:
            missing_room_type_count += 1
        elif clean not in ROOM_TYPE_TO_LABEL:
            unsupported_room_types.append(value)

    for edge in edges:
        value = _dictionary_value(edge, doorTypeKey, None)
        clean = _clean_string(value)
        if clean is None:
            missing_door_type_count += 1
        elif clean not in DOOR_TYPE_TO_CONNECTIVITY:
            unsupported_door_types.append(value)

    summary = {
        "vertex_count": len(vertices),
        "edge_count": len(edges),
        "missing_room_type_count": missing_room_type_count,
        "missing_door_type_count": missing_door_type_count,
        "unsupported_room_types": sorted(set(unsupported_room_types)),
        "unsupported_door_types": sorted(set(unsupported_door_types)),
    }

    if not silent:
        for key, value in summary.items():
            print(f"{key}: {value}")

    return summary


In [8]:
from pathlib import Path
from topologicpy.Graph import Graph

def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "01_Research_Dataset_and_References").exists():
            return candidate
    raise FileNotFoundError("Project root not found. Open the notebook from inside GraphML_Final_Assigment.")

PROJECT_ROOT = find_project_root()
GRAPH_PATH = (
    PROJECT_ROOT
    / "03_Build_Graph_Representation"
    / "graph_outputs"
    / "habitar72_room_graph.json"
)

graph = Graph.ByJSONPath(str(GRAPH_PATH))

print("Graph path:", GRAPH_PATH)
print("Graph loaded:", graph is not None)


Graph path: c:\Users\Arq. David Agudelo\OneDrive\Documentos\GitHub\GraphML_Final_Assigment\03_Build_Graph_Representation\graph_outputs\habitar72_room_graph.json
Graph loaded: True


In [9]:
summary = CheckMSDGraphPreparation(graph)

assert summary["vertex_count"] > 0
assert summary["edge_count"] > 0
assert summary["missing_room_type_count"] == 0
assert summary["missing_door_type_count"] == 0
assert summary["unsupported_room_types"] == []
assert summary["unsupported_door_types"] == []

print("Preparation validation: PASSED")

vertex_count: 35
edge_count: 39
missing_room_type_count: 0
missing_door_type_count: 0
unsupported_room_types: []
unsupported_door_types: []
Preparation validation: PASSED


In [10]:
# Encode MSD features and export the final CSV dataset.
graph = EncodeMSDGraphFeatures(graph)

DATASET_DIR = (
    PROJECT_ROOT
    / "05_Run_Node_Classification"
    / "ml_dataset"
)
DATASET_DIR.mkdir(parents=True, exist_ok=True)

export_status = Graph.ExportToCSV(
    graph,
    path=str(DATASET_DIR),
    nodeFeaturesKeys=NODE_ZONING_FEATURE_KEYS + NODE_CONNECTIVITY_FEATURE_KEYS,
    edgeFeaturesKeys=EDGE_FEATURE_KEYS,
    overwrite=True,
)

if export_status is not True:
    raise RuntimeError("Graph.ExportToCSV did not complete successfully.")

print("CSV export: PASSED")
print("Dataset folder:", DATASET_DIR)


CSV export: PASSED
Dataset folder: c:\Users\Arq. David Agudelo\OneDrive\Documentos\GitHub\GraphML_Final_Assigment\05_Run_Node_Classification\ml_dataset


## Phase 3 — Run and inspect the encoding



### Basic usage

Run the helper after constructing your graph and after assigning `room_type` dictionaries to vertices and `door_type` dictionaries to edges.


In [11]:

# Example usage:
# graph = EncodeMSDGraphFeatures(graph)

# Then export using your existing TopologicPy workflow, for example:
# Graph.ExportToCSV(graph, path="/path/to/export_folder")



### Inspecting the encoded dictionaries

The following optional checks print the dictionary of the first vertex and first edge, so you can verify that the feature keys were added before export.


In [12]:

# Optional inspection:
# vertices = Graph.Vertices(graph)
# edges = Graph.Edges(graph)
#
# if vertices:
#     print("First vertex dictionary:")
#     print(Topology.Dictionary(vertices[0]))
#
# if edges:
#     print("First edge dictionary:")
#     print(Topology.Dictionary(edges[0]))


## Phase 4 — Validate graph preparation



### Optional: stricter preparation checks

This cell can be used before encoding to find missing or unsupported dictionary values.
